In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

TRAIN_SZ = 0.8
VALID_SZ = 1.0 - TRAIN_SZ

# Read the data
train = pd.read_csv('../datasets/home-data-for-ml-course/train.csv', index_col='Id') 
test = pd.read_csv('../datasets/home-data-for-ml-course/test.csv', index_col='Id')

# Remove rows with missing target
train.dropna(axis=0, subset=['SalePrice'], inplace=True)

# Separate target from predictors
y = train.SalePrice
X = train.drop(['SalePrice'], axis=1)
X_test = test.copy()

# Drop columns with missing values
cols_with_missing = [col for col in X.columns if X[col].isnull().any()] 
X.drop(cols_with_missing, axis=1, inplace=True)
X_test.drop(cols_with_missing, axis=1, inplace=True)

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=TRAIN_SZ, test_size=VALID_SZ,
                                                      random_state=0)

In [2]:
X_train.head()

,MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
619,20,RL,11694,Pave,Reg,Lvl,AllPub,Inside,Gtl,NridgHt,...,108,0,0,260,0,0,7,2007,New,Partial
871,20,RL,6600,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,...,0,0,0,0,0,0,8,2009,WD,Normal
93,30,RL,13360,Pave,IR1,HLS,AllPub,Inside,Gtl,Crawfor,...,0,44,0,0,0,0,8,2009,WD,Normal
818,20,RL,13265,Pave,IR1,Lvl,AllPub,CulDSac,Gtl,Mitchel,...,59,0,0,0,0,0,7,2008,WD,Normal
303,20,RL,13704,Pave,IR1,Lvl,AllPub,Corner,Gtl,CollgCr,...,81,0,0,0,0,0,1,2006,WD,Normal


In [3]:
X_valid.head()

,MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
530,20,RL,32668,Pave,IR1,Lvl,AllPub,CulDSac,Gtl,Crawfor,...,0,200,0,0,0,0,3,2007,WD,Alloca
492,50,RL,9490,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,...,0,32,0,0,0,0,8,2006,WD,Normal
460,50,RL,7015,Pave,IR1,Bnk,AllPub,Corner,Gtl,BrkSide,...,0,248,0,0,0,0,7,2009,WD,Normal
280,60,RL,10005,Pave,Reg,Lvl,AllPub,Inside,Gtl,ClearCr,...,117,0,0,0,0,0,3,2008,WD,Normal
656,160,RM,1680,Pave,Reg,Lvl,AllPub,Inside,Gtl,BrDale,...,0,0,0,0,0,0,3,2010,WD,Family


In [4]:
X_test.head()

,MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
1461,20,RH,11622,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,...,0,0,0,120,0,0,6,2010,WD,Normal
1462,20,RL,14267,Pave,IR1,Lvl,AllPub,Corner,Gtl,NAmes,...,36,0,0,0,0,12500,6,2010,WD,Normal
1463,60,RL,13830,Pave,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,...,34,0,0,0,0,0,3,2010,WD,Normal
1464,60,RL,9978,Pave,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,...,36,0,0,0,0,0,6,2010,WD,Normal
1465,120,RL,5005,Pave,IR1,HLS,AllPub,Inside,Gtl,StoneBr,...,82,0,0,144,0,0,1,2010,WD,Normal


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [6]:
### Aproach #1: Drop columns with categorical data
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

drop_mae = score_dataset(drop_X_train, drop_X_valid, y_train, y_valid)
print(f"MAE from Approach 1 (Drop categorical variables): {drop_mae}")

MAE from Approach 1 (Drop categorical variables): 17837.82570776256


In [7]:
print("Unique values in 'Condition2' column in training data:", sorted(X_train['Condition2'].unique()))
print("Unique values in 'Condition2' column in validation data:", sorted(X_valid['Condition2'].unique()))

Unique values in 'Condition2' column in training data: ['Artery', 'Feedr', 'Norm', 'PosA', 'PosN', 'RRAe']
Unique values in 'Condition2' column in validation data: ['Artery', 'Feedr', 'Norm', 'PosN', 'RRAn', 'RRNn']


In [8]:
# Categorical columns in the training data
object_cols = [col for col in X_train.columns 
               if X_train[col].dtype == "object"]

# Columns that can be safely ordinal encoded
good_label_cols = [col for col in object_cols 
                   if set(X_valid[col]).issubset(set(X_train[col]))]
        
# Problematic columns that will be dropped from the dataset
bad_label_cols = list(set(object_cols)-set(good_label_cols))
        
print('Categorical columns that will be ordinal encoded:', good_label_cols)
print('Categorical columns that will be dropped from the dataset:', bad_label_cols)

Categorical columns that will be ordinal encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'PavedDrive', 'SaleType', 'SaleCondition']
Categorical columns that will be dropped from the dataset: ['Condition2', 'Functional', 'RoofMatl']


In [9]:
### Approach #2: Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder

# Drop categorical columns that will not be encoded
label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_valid = X_valid.drop(bad_label_cols, axis=1)

# Apply ordinal encoder
ordinal_encoder = OrdinalEncoder()
label_X_train[good_label_cols] = ordinal_encoder.fit_transform(label_X_train[good_label_cols])
label_X_valid[good_label_cols] = ordinal_encoder.transform(label_X_valid[good_label_cols])

label_mae = score_dataset(label_X_train, label_X_valid, y_train, y_valid)
print(f"MAE from Approach 2 (Ordinal Encoding): {label_mae}") 

MAE from Approach 2 (Ordinal Encoding): 17098.01649543379


In [10]:
# Get number of unique entries in each column with categorical data
object_nunique = list(map(lambda col: X_train[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

# Print number of unique entries by column, in ascending order
sorted(d.items(), key=lambda x: x[1])

[('Street', 2),
 ('Utilities', 2),
 ('CentralAir', 2),
 ('LandSlope', 3),
 ('PavedDrive', 3),
 ('LotShape', 4),
 ('LandContour', 4),
 ('ExterQual', 4),
 ('KitchenQual', 4),
 ('MSZoning', 5),
 ('LotConfig', 5),
 ('BldgType', 5),
 ('ExterCond', 5),
 ('HeatingQC', 5),
 ('Condition2', 6),
 ('RoofStyle', 6),
 ('Foundation', 6),
 ('Heating', 6),
 ('Functional', 6),
 ('SaleCondition', 6),
 ('RoofMatl', 7),
 ('HouseStyle', 8),
 ('Condition1', 9),
 ('SaleType', 9),
 ('Exterior1st', 15),
 ('Exterior2nd', 16),
 ('Neighborhood', 25)]

In [11]:
N = 10

high_cardinality_cols = len([_ for _ in d.items() if _[1] > N])
print(f"Categorical variables in the training data that have cardinality > {N}: {high_cardinality_cols}")

num_cols_neighborhood = d['Neighborhood']
print(f"Columns needed to One-Hot encode the 'Neighborhood' variable in the training data: {num_cols_neighborhood}")


Categorical variables in the training data that have cardinality > 10: 3
Columns needed to One-Hot encode the 'Neighborhood' variable in the training data: 25


In [12]:
# Columns that will be one-hot encoded
low_cardinality_cols = [col for col in object_cols if X_train[col].nunique() < N]

# Columns that will be dropped from the dataset
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols))

print(f"Categorical columns that will be one-hot encoded: {low_cardinality_cols}")
print(f"Categorical columns that will be dropped from the dataset: {high_cardinality_cols}")

Categorical columns that will be one-hot encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']
Categorical columns that will be dropped from the dataset: ['Exterior1st', 'Exterior2nd', 'Neighborhood']


In [13]:
# Approach #3: One-Hot encoding
from sklearn.preprocessing import OneHotEncoder

# Apply one-hot encoder to each column with categorical data
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[low_cardinality_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[low_cardinality_cols]))

# One-hot encoding removed index; put it back
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.copy().drop(object_cols, axis=1)
num_X_valid = X_valid.copy().drop(object_cols, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

oh_mae = score_dataset(OH_X_train, OH_X_valid, y_train, y_valid)
print(f"MAE from Approach 3 (One-Hot Encoding): {oh_mae}") 


MAE from Approach 3 (One-Hot Encoding): 17525.345719178084


In [14]:
# Remove row with missing values
X_test.dropna(axis=0, inplace=True)

# Drop categorical columns that will not be encoded
X_test.drop(bad_label_cols, axis=1, inplace=True)

# Apply ordinal encoder
X_test[good_label_cols] = ordinal_encoder.transform(X_test[good_label_cols])

# Define and fit model
model = RandomForestRegressor(n_estimators=250, random_state=1)
model.fit(label_X_train, y_train)

# Predict test values
y_test = model.predict(X_test)

output = pd.DataFrame({'Id': X_test.index, 'SalePrice': y_test})
# output.to_csv('submission.csv', index=False)

In [16]:
from IPython.display import display, HTML

report = f"""
<div style="display: flex;">
    <div style="border-right: #fcfcfc 1px solid;">
        {X_test.to_html(index=False)}
    </div>
    <div style="background-color: #f0f6fe;">
        {output.drop('Id', axis=1).to_html(index=False)}
    </div>
</div>
"""

display(HTML(report))

MSSubClass,MSZoning,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,Exterior1st,Exterior2nd,ExterQual,ExterCond,Foundation,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Fireplaces,GarageCars,GarageArea,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
20,2.0,11622,1.0,3.0,3.0,0.0,4.0,0.0,12.0,1.0,0.0,2.0,5,6,1961,1961,1.0,12.0,13.0,3.0,4.0,1.0,468.0,144.0,270.0,882.0,1.0,4.0,1.0,896,0,0,896,0.0,0.0,1,0,2,1,3.0,5,0,1.0,730.0,2.0,140,0,0,0,120,0,0,6,2010,8.0,4.0
20,3.0,14267,1.0,0.0,3.0,0.0,0.0,0.0,12.0,2.0,0.0,2.0,6,6,1958,1958,3.0,13.0,14.0,3.0,4.0,1.0,923.0,0.0,406.0,1329.0,1.0,4.0,1.0,1329,0,0,1329,0.0,0.0,1,1,3,1,2.0,6,0,1.0,312.0,2.0,393,36,0,0,0,0,12500,6,2010,8.0,4.0
60,3.0,13830,1.0,0.0,3.0,0.0,4.0,0.0,8.0,2.0,0.0,5.0,5,5,1997,1998,1.0,12.0,13.0,3.0,4.0,2.0,791.0,0.0,137.0,928.0,1.0,2.0,1.0,928,701,0,1629,0.0,0.0,2,1,3,1,3.0,6,1,2.0,482.0,2.0,212,34,0,0,0,0,0,3,2010,8.0,4.0
60,3.0,9978,1.0,0.0,3.0,0.0,4.0,0.0,8.0,2.0,0.0,5.0,6,6,1998,1998,1.0,12.0,13.0,3.0,4.0,2.0,602.0,0.0,324.0,926.0,1.0,0.0,1.0,926,678,0,1604,0.0,0.0,2,1,3,1,2.0,7,1,2.0,470.0,2.0,360,36,0,0,0,0,0,6,2010,8.0,4.0
120,3.0,5005,1.0,0.0,1.0,0.0,4.0,0.0,22.0,2.0,4.0,2.0,8,5,1992,1992,1.0,6.0,6.0,2.0,4.0,2.0,263.0,0.0,1017.0,1280.0,1.0,0.0,1.0,1280,0,0,1280,0.0,0.0,2,0,2,1,2.0,5,0,2.0,506.0,2.0,0,82,0,0,144,0,0,1,2010,8.0,4.0
60,3.0,10000,1.0,0.0,3.0,0.0,0.0,0.0,8.0,2.0,0.0,5.0,6,5,1993,1994,1.0,6.0,6.0,3.0,4.0,2.0,0.0,0.0,763.0,763.0,1.0,2.0,1.0,763,892,0,1655,0.0,0.0,2,1,3,1,3.0,7,1,2.0,440.0,2.0,157,84,0,0,0,0,0,4,2010,8.0,4.0
20,3.0,7980,1.0,0.0,3.0,0.0,4.0,0.0,8.0,2.0,0.0,2.0,6,7,1992,2007,1.0,6.0,6.0,3.0,2.0,2.0,935.0,0.0,233.0,1168.0,1.0,0.0,1.0,1187,0,0,1187,1.0,0.0,2,0,3,1,3.0,6,0,2.0,420.0,2.0,483,21,0,0,0,0,500,3,2010,8.0,4.0
60,3.0,8402,1.0,0.0,3.0,0.0,4.0,0.0,8.0,2.0,0.0,5.0,6,5,1998,1998,1.0,12.0,13.0,3.0,4.0,2.0,0.0,0.0,789.0,789.0,1.0,2.0,1.0,789,676,0,1465,0.0,0.0,2,1,3,1,3.0,7,1,2.0,393.0,2.0,0,75,0,0,0,0,0,5,2010,8.0,4.0
20,3.0,10176,1.0,3.0,3.0,0.0,4.0,0.0,8.0,2.0,0.0,2.0,7,5,1990,1990,1.0,6.0,6.0,3.0,4.0,2.0,637.0,0.0,663.0,1300.0,1.0,2.0,1.0,1341,0,0,1341,1.0,0.0,1,1,2,1,2.0,5,1,2.0,506.0,2.0,192,0,0,0,0,0,0,2,2010,8.0,4.0
20,3.0,8400,1.0,3.0,3.0,0.0,0.0,0.0,12.0,2.0,0.0,2.0,4,5,1970,1970,1.0,9.0,10.0,3.0,4.0,1.0,804.0,78.0,0.0,882.0,1.0,4.0,1.0,882,0,0,882,1.0,0.0,1,0,2,1,3.0,4,0,2.0,525.0,2.0,240,0,0,0,0,0,0,4,2010,8.0,4.0
